# Feature Significance Analysis

This notebook analyzes feature importance using multiple methods:
1. Correlation Analysis
2. Statistical Significance (p-values)
3. Random Forest Importance
4. RFE (Recursive Feature Elimination)
5. Lasso Feature Selection

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import RFE, SelectFromModel, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8')  # Use correct style name
sns.set_theme()  # Use seaborn's default theme

## Load and Prepare Data

In [ ]:
# Load data
data = pd.read_csv('../../data/processed/cleaned_train.csv')
X = data.drop('PremiumAmount', axis=1)
y = data['PremiumAmount']

print("Dataset shape:", data.shape)
print("First five entries of the dataset:")
data.head()

## 1. Correlation Analysis

In [ ]:
# Correlation matrix and heatmap
corr_matrix = data.corr()
target_corr = corr_matrix['PremiumAmount'].abs().sort_values(ascending=False)

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

print("\nCorrelations with PremiumAmount:")
print(target_corr)

## 2. Statistical Significance

In [ ]:
# Calculate p-values
p_values = pd.DataFrame(index=X.columns, columns=['p_value', 'correlation'])
for feature in X.columns:
    correlation, p_value = stats.pearsonr(X[feature], y)
    p_values.loc[feature] = [p_value, correlation]

p_values = p_values.sort_values('p_value')
print("Statistical Significance:")
print(p_values)

## 3. Random Forest Importance

In [ ]:
# Train Random Forest and get feature importances
model = RandomForestRegressor(random_state=42, n_jobs=-1)
model.fit(X, y)

importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=importances, x='Importance', y='Feature')
plt.title('Random Forest Feature Importance')
plt.show()

## Harmful Feature Detection

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Combine all metrics for harmful feature detection
harmful_features = pd.DataFrame({
    'Feature': X.columns,
    'P_Value': p_values['p_value'],
    'RF_Importance': model.feature_importances_.set_index('Feature')['Importance'],
    'VIF': vif_data.set_index('Feature')['VIF']
})

# Flag potentially harmful features
harmful_features['Is_Harmful'] = (
    (harmful_features['P_Value'] > 0.05) |  # Not statistically significant
    (harmful_features['RF_Importance'] < 0.01) |  # Very low importance
    (harmful_features['VIF'] > 5)  # High multicollinearity
)

# Sort by potential harm (highest VIF and p-value, lowest importance)
harmful_features['Harm_Score'] = (
    harmful_features['P_Value'] + 
    harmful_features['VIF']/10 - 
    harmful_features['RF_Importance']
)

harmful_features = harmful_features.sort_values('Harm_Score', ascending=False)

print("Potentially Harmful Features:")
display(harmful_features[harmful_features['Is_Harmful']])

# Visualize harmful features
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=harmful_features,
    x='RF_Importance',
    y='P_Value',
    size='VIF',
    hue='Is_Harmful',
    alpha=0.6
)
plt.axhline(y=0.05, color='r', linestyle='--', alpha=0.3)
plt.axvline(x=0.01, color='r', linestyle='--', alpha=0.3)
plt.title('Feature Harm Analysis')
plt.show()

## Combined Feature Analysis

In [ ]:
# Combine all metrics
feature_analysis = pd.DataFrame({
    'Feature': X.columns,
    'Correlation': abs(target_corr[X.columns]),
    'P_Value': p_values['p_value'],
    'RF_Importance': importances.set_index('Feature')['Importance']
})

# Calculate combined score
feature_analysis['Combined_Score'] = (
    feature_analysis['Correlation'] * 0.3 +
    feature_analysis['RF_Importance'] * 0.5 +
    (1 - feature_analysis['P_Value']) * 0.2
)

# Sort and display results
feature_analysis = feature_analysis.sort_values('Combined_Score', ascending=False)
print("Final Feature Rankings:")
display(feature_analysis)

# Save results
feature_analysis.to_csv('../data/feature_analysis.csv', index=False)

# Plot final results
plt.figure(figsize=(15, 10))
sns.heatmap(
    feature_analysis.set_index('Feature'),
    annot=True,
    cmap='YlOrRd',
    fmt='.3f'
)
plt.title('Feature Significance Metrics')
plt.tight_layout()
plt.show()